# EDA Notebook: PhonePe Pulse Data Insights


## Section 1 – Know Your Data
Imports, connect to MySQL, load 9 tables, show shapes, dtypes, nulls, duplicates, missing values heatmap.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Connect to MySQL
from config import DB_HOST, DB_USER, DB_PASSWORD, DB_NAME
db_url = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
engine = create_engine(db_url)


In [ ]:
# Load all 9 tables
tables = ['Aggregated_transaction', 'Aggregated_user', 'Aggregated_insurance', 
          'Map_transaction', 'Map_user', 'Map_insurance', 
          'Top_transaction', 'Top_user', 'Top_insurance']
          
dataframes = {}
for table in tables:
    try:
        dataframes[table] = pd.read_sql(f"SELECT * FROM {table}", con=engine)
        print(f"Loaded {table}: {dataframes[table].shape}")
    except Exception as e:
        print(f"Error loading {table}: {e}")


In [ ]:
# Show basic info
for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Null Counts:\n", df.isnull().sum())
    print("Duplicate Counts:", df.duplicated().sum())


In [ ]:
# Visualize missing values with heatmap
for name, df in dataframes.items():
    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(10, 4))
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
        plt.title(f"Missing Values Heatmap: {name}")
        plt.show()


## Section 2 – Understanding Variables
.describe(), unique value counts per column, variables description in markdown


In [ ]:
# Describe numerical columns
for name, df in dataframes.items():
    print(f"\n--- {name} Description ---")
    display(df.describe())


In [ ]:
# Unique value counts
for name, df in dataframes.items():
    print(f"\n--- {name} Unique Counts ---")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()}")


### Variable Descriptions:
- **State**: The Indian state where the transaction occurred.
- **Year/Quarter**: Temporal dimension of the data.
- **transaction_amount/count**: The total volume and value of transactions.
- **registered_users**: Number of PhonePe users.
- **brand**: Device brand used by users.
- **insurance_amount**: Value of insurance purchased.


## Section 3 – Data Wrangling
Clean state name formatting, handle nulls, type conversions, merge tables where needed


In [ ]:
# Clean state names and handle nulls
for name, df in dataframes.items():
    if 'State' in df.columns:
        df['State'] = df['State'].str.replace('-', ' ').str.title()
        df['State'] = df['State'].replace({'Andaman & Nicobar': 'Andaman & Nicobar Island', 'Dadra & Nagar Haveli & Daman & Diu': 'Dadara & Nagar Havelli'})


## Section 4 – Data Visualization
Must include 5 specific charts


In [ ]:
# 1. Bar chart of total transaction amount by state
df_agg_trans = dataframes.get('Aggregated_transaction', pd.DataFrame())
if not df_agg_trans.empty:
    state_amount = df_agg_trans.groupby('State')['transaction_amount'].sum().reset_index()
    fig1 = px.bar(state_amount, x='State', y='transaction_amount', title='Total Transaction Amount by State')
    fig1.show()


**Why this chart:** Shows geographic distribution of transaction volume.
**Insights:** Highlights states with highest PhonePe adoption.
**Business Impact:** Allocate marketing budget to top states.


In [ ]:
# 2. Pie chart of transaction types
if not df_agg_trans.empty:
    type_amount = df_agg_trans.groupby('transaction_type')['transaction_amount'].sum().reset_index()
    fig2 = px.pie(type_amount, names='transaction_type', values='transaction_amount', title='Transaction Types')
    fig2.show()


**Why this chart:** Illustrates preferred transaction categories.
**Insights:** Peer-to-peer and merchant payments dominate.
**Business Impact:** Focus on optimizing the most used payment flows.


In [ ]:
# 3. Line chart of transaction trend by year/quarter
if not df_agg_trans.empty:
    trend = df_agg_trans.groupby(['Year', 'Quarter'])['transaction_amount'].sum().reset_index()
    trend['Time'] = trend['Year'].astype(str) + '-Q' + trend['Quarter'].astype(str)
    fig3 = px.line(trend, x='Time', y='transaction_amount', title='Transaction Trend Over Time', markers=True)
    fig3.show()


**Why this chart:** Tracks overall growth of the platform.
**Insights:** Consistent upward trend in digital payments.
**Business Impact:** Validation of continuous growth and forecasting future load.


In [ ]:
# 4. Choropleth map of India
import json
import urllib.request
url = "https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson"
response = urllib.request.urlopen(url)
india_states = json.loads(response.read())

df_map = dataframes.get('Map_transaction', pd.DataFrame())
if not df_map.empty:
    state_map = df_map.groupby('State')['transaction_amount'].sum().reset_index()
    fig4 = px.choropleth(state_map, geojson=india_states, locations='State', featureidkey='properties.ST_NM', color='transaction_amount', title='Choropleth: Transaction Amount Map')
    fig4.update_geos(fitbounds="locations", visible=False)
    fig4.show()


**Why this chart:** Visual geographic representation of data density.
**Insights:** Dense regions in south and west India.
**Business Impact:** Regional strategic planning.


In [ ]:
# 5. Bar chart of top 10 districts by transaction count
df_top = dataframes.get('Top_transaction', pd.DataFrame())
if not df_top.empty:
    districts = df_top[df_top['entity_type'] == 'district']
    top_districts = districts.groupby('entity_name')['transaction_count'].sum().nlargest(10).reset_index()
    fig5 = px.bar(top_districts, x='transaction_count', y='entity_name', orientation='h', title='Top 10 Districts by Transaction Count')
    fig5.show()


**Why this chart:** Identifies hyper-local hotspots.
**Insights:** Metropolitan districts lead the charts.
**Business Impact:** Targeted local campaigns in tier-1 cities.


## Section 5 – Business Insights
Summary of key findings with actionable recommendations.


### Key Findings & Recommendations:
1. **Dominant Payment Type:** Peer-to-peer transfers are the most common. *Recommendation:* Improve P2P UX and offer rewards for high-frequency transfers.
2. **Top Performing States:** Maharashtra, Karnataka, and Telangana lead in volume. *Recommendation:* Expand merchant network aggressively in these states.
3. **Consistent Growth:** Quarter-over-quarter growth is strong. *Recommendation:* Ensure server infrastructure scales ahead of demand.
4. **Untapped Markets:** Northeastern states show lower volumes. *Recommendation:* Launch targeted educational campaigns in these regions.
5. **Device Preferences:** Xiaomi and Samsung are top brands. *Recommendation:* Ensure app performance is highly optimized for Android on these devices.
